## Find Kelvin Waves Events

### Import pacakges

In [1]:
import numpy as np
import netCDF4 as nc
import pandas as pd

from datetime import datetime
from typing import Dict, List, Tuple
from pathlib import Path

from tqdm import tqdm

from matplotlib import pyplot as plt

### Load data

In [2]:
# path directory
file_path: Path = Path("/work/DATA/Satellite/OLR/olr_anomaly.nc")

# Load anomalous OLR dataset
with nc.Dataset(file_path, "r") as olr_ds:
    
    ## Load coordinates
    coords: Dict[str, np.ndarray] = {
        key: olr_ds.variables[key][...]
        for key in olr_ds.dimensions.keys()
    }
    
    ## Setup up time limit
    time_range = pd.date_range(start="1979-01-01", periods=coords["time"].size, freq="D")

    time_limit = (time_range.year >= 2006) & (time_range.year <= 2017)

    ## Setup up spatial limit
    lat_limit: np.ndarray = (coords["lat"] >= -5.0) & (coords["lat"] <= 5.0)

    ## Load anomalous OLR
    olr: np.ndarray = olr_ds["olr"][time_limit, lat_limit, :]

olr -= np.nanmean(olr, axis=(0, 2), keepdims=True)

### Calculate symmetric data

In [3]:
olr_symm: np.ndarray = (olr + np.flip(olr)) / 2

olr_avg: np.ndarray = np.nansum(olr_symm * np.cos(np.radians(coords["lat"][lat_limit]))[None, :, None], axis=1)

ntime, nlon = olr_avg.shape

### Filtered

In [4]:
# generate wavenumber and frequency arrays
wnum: np.ndarray = np.fft.fftfreq(nlon, d=1/nlon)
freq: np.ndarray = np.fft.fftfreq(ntime, d=1)

wnums, freqs = np.meshgrid(wnum, freq)

# dispersion relation
kw_disp = lambda ed, k: np.sqrt(9.80665*ed) * 86400.0/(2*np.pi*6.371e6) * k

# perform 2D FFT to assigned grid
olr_fft: np.ndarray = np.fft.ifft(np.fft.fft(olr_avg, axis=1) / nlon, axis=0)

# Bandpass filter
target_windows: List[Tuple[int, ...]] = [
    (1, 3), (3, 5), (5, 7), (7, 9), (9, 11), (11, 13),
    (1, 13), (3, 8), (3, 11)
]

for i in tqdm(range(len(target_windows))):
    k0, k1 = target_windows[i]

    ## setup the targeting windows
    mask: Tuple = np.where(
        ((wnums >= k0) & (wnums <= k1) & 
        (freqs <= 1/2.5) & (freqs >= 1/30) &
        (freqs >= kw_disp(8, wnum)) & (freqs <= kw_disp(90, wnum)) ) |
        (
            (wnums <= -k0) & (wnums >= -k1) & 
        (freqs >= -1/2.5) & (freqs <= -1/30) &
        (freqs <= kw_disp(8, wnum)) & (freqs >= kw_disp(90, wnum)) 
        )
    )

    olr_fft_filtered: np.ndarray = np.zeros_like(olr_fft)
    olr_fft_filtered[mask] = olr_fft[mask]

    olr_filtered: np.ndarray = np.fft.fft(np.fft.ifft(olr_fft_filtered, axis=1)*nlon, axis=0)

    criteria: float = float(np.real(np.nanmean(olr_filtered) - 2.75 * np.nanstd(olr_filtered)))

    events: np.ndarray = np.array(np.where(olr_filtered <= criteria))

    # collect the tuple to original longitude values
    events_lon: np.ndarray = np.array([
        coords["lon"][events[1, i]]
        for i in range(events.shape[1])
    ])

    events_output: np.ndarray = np.zeros_like(events)
    events_output[0, :] = events[0]
    events_output[1, :] = events_lon
    
    np.savetxt(f"/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/KW_events/{target_windows[i]}.txt", events_output)

100%|██████████| 9/9 [00:02<00:00,  3.99it/s]
